<a href="https://colab.research.google.com/github/madhumitha006/Madhumitha-Codeboosters-internship-2026/blob/main/Day_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LLM are trained using next token prediction they learn to predict the most statically likely next word they  do not store or look up facts like a database

n_results =3 means return the 3 most similar chunks

chunk - A small piece of a document

embedding - a vector representing meaning

vector database - A database that stores and searches vector by similarly

retrival - finding the most relevant chunks for a glam query

constant injection - adding retrived chunks into the llms prompt

In [25]:
!pip install sentence-transformers chromadb groq pandas -q

In [26]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
print("All libraries imported succefully.")
print("Ready to build a RAG system.")

All libraries imported succefully.
Ready to build a RAG system.


In [27]:
import os
GROQ_API_KEY ="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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized:")
print("Note: If you see an authentication error later, double-check your API key")

Groq API client initialized:
Note: If you see an authentication error later, double-check your API key


In [28]:
df = pd.read_csv('college_notes.csv')
print("Shape of dataset:",df.shape)
print("\nColumn names:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [29]:
print("Subjects in this dataset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id', 'subject', 'topic']].to_string(index=False))
print("\nLength of content (number of characters) for each note:")
print(df['content'].str.len())

Subjects in this dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Pytho

In [30]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject":row['subject'], "topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared:{len(documents)}")
print(f"First document ID    :{ids[0]}")
print(f"First metadata       :{metadatas[0]}")
print(f"First 100 chars of doc:{documents[0][:100]}...")

Total chunks prepared:15
First document ID    :note_N001
First metadata       :{'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [31]:
print("Loading embedding model...")
print("(This may take 30-60 seconds on first run - model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\nEmbedding model loaded successfully.")
test_embedding = embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape:{test_embedding.shape}")

Loading embedding model...
(This may take 30-60 seconds on first run - model is being downloaded)
(Subsequent runs will be faster as the model is cached)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded successfully.
Test embedding shape:(384,)


In [32]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds...")

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print(f"\nEmbedding matrix shape: {embeddings.shape}")

embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    metadatas=metadatas,
    ids=ids
)

print("\nDocuments successfully added to ChromaDB.")
print(f"Total documents in collection: {collection.count()}")

Generating embeddings for all 15 notes...
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)

Documents successfully added to ChromaDB.
Total documents in collection: 15


In [35]:
def retrieve_relevant_chunks(question, top_k=3):
    """
    Given a user question, retrieve the most relevant document chunks from ChromaDB.

    Parameters:
        question (str): The user's question as a text string.
        top_k (int): Number of top results to return (default: 3).

    Returns:
        dict: A dictionary containing the retrieved documents,
              metadata, distances, and IDs.
    """
    question_embedding = embedding_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )
    return results
print("Retrieval function defined successfully.")
print("Function:retrieve_relavant_chunks(question,top_k=3)")

Retrieval function defined successfully.
Function:retrieve_relavant_chunks(question,top_k=3)


In [36]:
test_question = "What is ETL and how does it work in data engineering"

print(f"The Question: {test_question}")
print("=" * 60)

results = retrieve_relevant_chunks(test_question, top_k=3)

print("\nTop 3 Retrieved Chunks:")
print("=" * 60)

for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"\nResult {i+1}:")
    print(f"  Subject : {meta['subject']}")
    print(f"  Topic   : {meta['topic']}")
    print(f"  Distance: {dist:.4f}")
    print(f"  Context : {doc[:120]}...")

The Question: What is ETL and how does it work in data engineering

Top 3 Retrieved Chunks:

Result 1:
  Subject : Data Engineering
  Topic   : ETL Pipelines
  Distance: 0.2041
  Context : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2:
  Subject : Data Engineering
  Topic   : APIs and Data Collection
  Distance: 1.1100
  Context : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3:
  Subject : Python Programming
  Topic   : Data Visualization
  Distance: 1.3892
  Context : Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


In [38]:
def build_context_from_results(results):
  context_parts =[]
  for i, (doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
      )):
    chunk_text = f"[Source {i+1}: {meta['subject']}-{meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  context_str = "\n\n--\n\n".join(context_parts)
  return context_str
context = build_context_from_results(results)
print("Build context string from retrieved chunks:")
print("=" * 60)
print(context)
print(f"\nTotal context length:{len(context)} characters")

Build context string from retrieved chunks:
[Source 1: Data Engineering-ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

--

[Source 2: Data Engineering-APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.

--

[Source 3: Python Programming-Data Visualization]
Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts line plots histograms and pie charts that help humans understand patterns in data.

Total context length:848 characters


In [39]:
def generate_rag_answer(question, context):
    system_prompt = """
You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base and a student's question.

RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context, say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner-friendly.
5. Mention which source the information came from when possible.
"""

    user_prompt = f"""
Context from Knowledge Base:

{context}

---

Student's Question: {question}

Please answer the question based only on the context provided above.
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content
    return answer


print("RAG generation function defined.")

RAG generation function defined.


In [40]:
def ask_college_assistant(question,top_k=3,verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("=" * 60)
    print("Step 1: Retrieving most relevant chunks...")
  results = retrieve_relevant_chunks(question,top_k=top_k)
  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base.")
    for i,meta in enumerate(results['metadatas'][0]):
      print(f"\nResult {i+1}.{meta['subject']} - {meta['topic']}")
    print("\nStep 2:Building context string ...")
    context = build_context_from_results(results)
  if verbose:
    print(f"Context built ({len(context)}) characters")
    print("\nStep 3:Sending to LLM for answer generation...")
  answer = generate_rag_answer(question,context)
  if verbose:
    print("\n" + "=" * 60)
    print("ANSWER:")
    print("=" * 60)
    print(answer)
    print("=" * 60)
  return answer
print("Complete RAG pipeline function ready.")
print("Function:ask_college_assistant(question,top_k=3)")


Complete RAG pipeline function ready.
Function:ask_college_assistant(question,top_k=3)


In [41]:
question_1 = "What is ETL and what are its three main stages?"
answer_1 = ask_college_assistant(question_1,top_k=3,verbose=True)

Question: What is ETL and what are its three main stages?
Step 1: Retrieving most relevant chunks...
Retrieved 3 chunks from the knowledge base.

Result 1.Data Engineering - ETL Pipelines

Result 2.Generative AI - Retrieval Augmented Generation

Result 3.Generative AI - Prompt Engineering

Step 2:Building context string ...
Context built (922) characters

Step 3:Sending to LLM for answer generation...

ANSWER:
According to Source 1: Data Engineering-ETL Pipelines, ETL stands for Extract Transform Load. The three main stages of ETL are:

1. Extract: collecting raw data from different sources
2. Transform: transforming the raw data into a clean and structured format
3. Load: loading the transformed data into a database or data warehouse for analysis.

This information is found in Source 1: Data Engineering-ETL Pipelines.


In [43]:
question_2 = "How do embeddings help in building search systems"
answer_2 = ask_college_assistant(question_2,top_k=3, verbose=True)

Question: How do embeddings help in building search systems
Step 1: Retrieving most relevant chunks...
Retrieved 3 chunks from the knowledge base.

Result 1.Generative AI - Retrieval Augmented Generation

Result 2.Generative AI - Large Language Models

Result 3.Machine Learning - Feature Engineering

Step 2:Building context string ...
Context built (902) characters

Step 3:Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.


In [42]:
question_3 = "What is the population of Tokyo"
print("Testing with an out of scope question (not in college notes)")
answer_3 = ask_college_assistant(question_3, top_k=3, verbose=True)

Testing with an out of scope question (not in college notes)
Question: What is the population of Tokyo
Step 1: Retrieving most relevant chunks...
Retrieved 3 chunks from the knowledge base.

Result 1.Generative AI - Large Language Models

Result 2.Data Engineering - SQL Databases

Result 3.Data Engineering - Data Cleaning

Step 2:Building context string ...
Context built (791) characters

Step 3:Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.


In [49]:
def retrive_key_subject(question,subject_filter,top_k=3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
      where={"subject": subject_filter}
  )
  return results
print("Retrieving only from GenAI subject:")
print("=" * 50)
filtered_results = retrive_key_subject(
    question="How do LLMs generate text?",
    subject_filter="GenAI",
    top_k=3
)
for i, (doc, dist) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
)):
    print(f"Result{i+1}:[{meta['subject']}]{meta['topic']}")
    print(f"{doc[:100]}...")

Retrieving only from GenAI subject:


In [50]:
# Install libraries
!pip install chromadb sentence-transformers groq pandas -q

In [51]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

In [52]:
df = pd.read_csv("college_notes.csv")

print(df.head())

  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N005  Data Engineering      Big Data and PySpark   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Big Data refers to extremely large datasets th...  


In [54]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="college_notes"
)

In [56]:
documents = df["content"].tolist()

ids = [
    f"note_{row['note_id']}"
    for row in df.to_dict("records")
]

metadatas = [
    {
        "subject": row["subject"],
        "topic": row["topic"]
    }
    for row in df.to_dict("records")
]

In [57]:
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
    ids=ids
)

print("Documents added successfully.")
print("Total documents:", collection.count())

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Documents added successfully.
Total documents: 15


In [58]:
def retrieve_relevant_chunks(
    question,
    top_k=3
):
    query_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results

In [59]:
groq_client = Groq(
    api_key="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"
)

In [60]:
def generate_rag_answer(
    question,
    context
):

    system_prompt = """
You are a helpful academic assistant.

Answer ONLY using the context.

If the answer is not found, say:
I don't have enough information in my knowledge base to answer this question.
"""

    user_prompt = f"""
Context:

{context}

Question:
{question}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.1,
        max_tokens=300
    )

    return response.choices[0].message.content

In [61]:
def ask_college_assistant(
    question,
    top_k=3,
    verbose=True
):

    results = retrieve_relevant_chunks(
        question,
        top_k
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    answer = generate_rag_answer(
        question,
        context
    )

    if verbose:
        print("\nQuestion:")
        print(question)

        print("\nRetrieved Sources:")

        for i, meta in enumerate(
            results["metadatas"][0]
        ):
            print(
                f"{i+1}. {meta['subject']} - {meta['topic']}"
            )

        print("\nAnswer:")
        print(answer)

    return answer

In [62]:
question = "What is ETL and what are its three main stages?"

answer = ask_college_assistant(
    question,
    top_k=3,
    verbose=True
)


Question:
What is ETL and what are its three main stages?

Retrieved Sources:
1. Data Engineering - ETL Pipelines
2. Generative AI - Retrieval Augmented Generation
3. Generative AI - Prompt Engineering

Answer:
ETL stands for Extract Transform Load. The three main stages of ETL are: 

1. Extract: collecting raw data from different sources
2. Transform: transforming the data into a clean and structured format
3. Load: loading the transformed data into a database or data warehouse for analysis.
